In [7]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession, Row
from pyspark.ml.linalg import Vectors
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Stop any existing SparkContext if it's running to avoid conflicts
# This is crucial to prevent 'Cannot run multiple SparkContexts at once' errors.
if 'SpContext' in globals() and SpContext:
    SpContext.stop()

# Create a SparkContext
conf = SparkConf().setMaster("local").setAppName("RandomForestLiverDisorder")
SpContext = SparkContext(conf = conf)

# Create a SparkSession
SpSession = SparkSession.builder.getOrCreate()

print("=== Spark context and session initialized for Liver Disorder task ===")

=== Spark context and session initialized for Liver Disorder task ===


In [9]:
!wget -O 9-liver-disorders-testing.txt https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/9-liver-disorders-testing.txt

--2026-04-07 01:20:59--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/9-liver-disorders-testing.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6307 (6.2K) [text/plain]
Saving to: ‘9-liver-disorders-testing.txt’

9-liver-disorders-t 100%[===================>]   6.16K  --.-KB/s    in 0s      

2026-04-07 01:20:59 (51.7 MB/s) - ‘9-liver-disorders-testing.txt’ saved [6307/6307]



In [10]:
# Load the CSV file into an RDD
liverData = SpContext.textFile("9-liver-disorders-testing.txt")
liverData.cache()
print(f"Total lines in dataset: {liverData.count()}")

Total lines in dataset: 100


In [12]:
def transformToNumericLiver(inputStr):
    # Dataset format from 9-liver-disorders-testing.txt appears to be LIBSVM format:
    # outcome feature1:value1 feature2:value2 ...
    # User specified: MCV, ALP, ALT, AST, GGT as features
    # Outcome: 0.0 (no disorder), 1.0 (has disorder)

    parts = inputStr.strip().split(" ")
    outcome = float(parts[0]) # The first part is the label/outcome

    # Initialize features
    mcv, alp, alt, ast, ggt = 0.0, 0.0, 0.0, 0.0, 0.0

    # Parse feature:value pairs
    for item in parts[1:]:
        if ":" in item:
            idx, val = item.split(":")
            feature_idx = int(idx)
            feature_val = float(val)

            # Map to named features based on original problem description
            # 1. mean corpuscular volume (MCV value)
            # 2. alkphos alkaline phosphotase (ALP value)
            # 3. alanine aminotransferase (ALT value)
            # 4. aspartate aminotransferase (AST value)
            # 5. gammagt gamma-glutamyl transpeptidase (GGT value)
            if feature_idx == 1:
                mcv = feature_val
            elif feature_idx == 2:
                alp = feature_val
            elif feature_idx == 3:
                alt = feature_val
            elif feature_idx == 4:
                ast = feature_val
            elif feature_idx == 5:
                ggt = feature_val

    values = Row(
        OUTCOME=outcome, # outcome is already 0.0 or 1.0
        MCV=mcv,
        ALP=alp,
        ALT=alt,
        AST=ast,
        GGT=ggt
    )
    return values

# Change to a Vector of Rows
liverRows = liverData.map(transformToNumericLiver)
liverDf = SpSession.createDataFrame(liverRows)
liverDf.cache()
print("=== Cleaned data sample ===")
liverDf.show(5)

=== Cleaned data sample ===
+-------+----+----+----+----+----+
|OUTCOME| MCV| ALP| ALT| AST| GGT|
+-------+----+----+----+----+----+
|    0.0|85.0|64.0|59.0|32.0|23.0|
|    0.0|86.0|54.0|33.0|16.0|54.0|
|    0.0|91.0|78.0|34.0|24.0|36.0|
|    0.0|87.0|70.0|12.0|28.0|10.0|
|    0.0|98.0|55.0|13.0|17.0|17.0|
+-------+----+----+----+----+----+
only showing top 5 rows


In [13]:
print("=== Basic statistical analysis on features ===")
liverDf.describe().show()

print("=== Correlation to OUTCOME for features ===")
for i in liverDf.columns:
    if not( isinstance(liverDf.select(i).take(1)[0][0], str)) :
      print( "Correlation to OUTCOME for ", i, \
      liverDf.stat.corr('OUTCOME',i))

=== Basic statistical analysis on features ===
+-------+------------------+----------------+------------------+-----------------+-----------------+------------------+
|summary|           OUTCOME|             MCV|               ALP|              ALT|              AST|               GGT|
+-------+------------------+----------------+------------------+-----------------+-----------------+------------------+
|  count|               100|             100|               100|              100|              100|               100|
|   mean|              0.44|           89.64|             67.24|            27.89|            24.75|             38.79|
| stddev|0.4988876515698587|4.29357048093396|17.582773939112663|20.79038967961647|9.331574509603556|41.384693164053154|
|    min|               0.0|            79.0|              37.0|              4.0|              8.0|               8.0|
|    max|               1.0|           103.0|             134.0|            155.0|             68.0|             

In [14]:
def transformToLabeledPointLiverML(row):
    lp = (row["OUTCOME"],
          Vectors.dense([
              row["MCV"],
              row["ALP"],
              row["ALT"],
              row["AST"],
              row["GGT"]
          ]))
    return lp

liverLp = liverDf.rdd.map(transformToLabeledPointLiverML)
liverLpDf = SpSession.createDataFrame(liverLp, ["label", "features"])
liverLpDf.cache()
print("=== LabeledPoint DataFrame sample ===")
liverLpDf.select("label", "features").show(5)

=== LabeledPoint DataFrame sample ===
+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|[85.0,64.0,59.0,3...|
|  0.0|[86.0,54.0,33.0,1...|
|  0.0|[91.0,78.0,34.0,2...|
|  0.0|[87.0,70.0,12.0,2...|
|  0.0|[98.0,55.0,13.0,1...|
+-----+--------------------+
only showing top 5 rows


In [15]:
# Split into training and testing data (80% training, 20% testing)
(trainingData, testData) = liverLpDf.randomSplit([0.8, 0.2], seed=42)

print(f"Training data count: {trainingData.count()}")
print(f"Test data count: {testData.count()}")

# Create the Random Forest model
# Using numTrees=10 as a reasonable default, adjust as needed
rfClassifier = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=10)
rfModel = rfClassifier.fit(trainingData)

print("=== Random Forest Model Training Complete ===")

Training data count: 83
Test data count: 17
=== Random Forest Model Training Complete ===


In [16]:
# Predict on the test data
predictions = rfModel.transform(testData)

# Evaluate precision/accuracy
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print("=== Model Evaluation Results ===")
print('Test Accuracy = ' + str(accuracy))
print('Test Error Rate = ' + str(1 - accuracy))

print("\n=== Confusion Matrix ===")
predictions.groupBy("label", "prediction").count().show()

=== Model Evaluation Results ===
Test Accuracy = 0.47058823529411764
Test Error Rate = 0.5294117647058824

=== Confusion Matrix ===
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0|    1|
|  0.0|       1.0|    7|
|  1.0|       0.0|    2|
|  0.0|       0.0|    7|
+-----+----------+-----+

